# ESCM 2.0 — Work package 2: causal ablations and counter-free inhibition

**Question.** Which parts of the original ESCM are necessary for autonomous firing and replacement of an old active location by a newly stimulated neighbor? Can a conventional decaying inhibitory trace replace the original spike counter **without losing those behaviors**?

We start from the established ESCM 1.x baseline, preserving its Brian2 scheduling, integration method, clock, and input-spike sequences. Four changes are made **one at a time**:

| Condition | Difference from baseline | What it isolates |
|---|---|---|
| Baseline | None | Reference behavior |
| No self-excitation | Set \(W_i=0\) | Need for recurrent self-feedback |
| No inhibition | Set \(W_k=0\) | Need for neighbor-mediated suppression |
| No persistence gate | Remove \(P\) from both membrane and inhibitory updates | Role of the activity gate |
| Synaptic inhibition | Replace counter-dependent assignment with spike-triggered inhibitory increments | First counter-free ESCM 2.0 candidate |

**Important:** The final condition deliberately changes the inhibitory dynamics; it is not expected to be numerically equivalent. No parameters are retuned yet, so failures are informative but do not rule out the proposed mechanism.

We keep a 5 × 5 recurrent network even in the single-pixel test: the legacy self-excitation is generated through *outgoing recurrent pathways*, so a literal single-neuron network without any recurrent connections would not reproduce that mechanism.

## 1. Controlled stimuli and fixed parameters

We reuse the four synthetic inputs from notebook 05:

- **Single:** one camera spike at the center (1 ms).
- **Repeated:** three center spikes (1, 5, 9 ms).
- **Sequential:** center then right neighbor (1, 5 ms).
- **Simultaneous:** center and right neighbor (1 ms).

All timestamps are aligned to the 0.5 ms Brian clock. The simulation continues to 60 ms. The output remains a spike train: a retained \(P\) value alone is **not** evidence of persistence.

The baseline parameters are from the existing script/notebooks, *not* the different parameter values reported in the paper.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from brian2 import *

prefs.codegen.target = "numpy"  # Easy-to-inspect reference backend
width = height = 5
N = width * height
center, neighbor = 2 * width + 2, 2 * width + 3
tau, clock_step = 0.2 * ms, 0.5 * ms
vt, vr = 0.1, 0.0
beta, Wi, Wk = 0.5, 6.0, -3.0
k, end_ms = 8, 60.0

stimuli = {
    "single": (np.array([center]), np.array([1.0])),
    "repeated": (np.array([center, center, center]), np.array([1.0, 5.0, 9.0])),
    "sequential": (np.array([center, neighbor]), np.array([1.0, 5.0])),
    "simultaneous": (np.array([center, neighbor]), np.array([1.0, 1.0])),
}
variants = (
    "baseline",
    "no_self_excitation",
    "no_inhibition",
    "no_gate",
    "synaptic_inhibition",
)
print("Fixed input protocols:", ", ".join(stimuli))
print(f"Network {width}x{height}, clock={clock_step}, tau={tau}")
print("Variants:", ", ".join(variants))

## 2. Implement one-variable-at-a-time experiments

The original membrane and synaptic states evolve as

\[
\tau\dot v=-v+\mathrm{ExtIn}+P(\mathrm{Exc}+\mathrm{Inh}),\qquad
\tau\dot z=-z\quad(z\in\{\mathrm{ExtIn},\mathrm{Exc},\mathrm{Inh}\}).
\]

In the **baseline**, a recurrent spike executes two ordered pathways:

\[
C_i\leftarrow C_i+1,\quad \mathrm{Exc}_{\mathrm{source}}\leftarrow W_i,
\quad \mathrm{Inh}_i\leftarrow P_iW_k C_i.
\]

The counter resets when the postsynaptic neuron spikes. A counter can therefore retain history from earlier presynaptic spikes.

In the **synaptic-inhibition candidate**, the counter is not used. Each neighboring spike instead adds a negative increment to the decaying inhibitory trace:

\[
\mathrm{Inh}_i\leftarrow\mathrm{Inh}_i+P_iW_k.
\]

This candidate **retains the existing gate** \(P\) and self-excitation. We therefore test only the effect of changing inhibition. The gate is present in both the inhibitory update and membrane equation, exactly as in the reference formulation; we are *not* yet proposing that this is the optimal biological model.

For the **no-gate ablation**, the persistence events remain active solely for monitoring, but \(P\) is removed from both the membrane equation and inhibition rule. The monitored \(P\) trace has no causal effect in that variant.

In [ ]:
def simulate(kind, input_ids, input_times_ms):
    start_scope()
    defaultclock.dt = clock_step

    params = {
        "tau": tau, "vt": vt, "vr": vr, "beta": beta,
        "Wi": 0.0 if kind == "no_self_excitation" else Wi,
        "Wk": 0.0 if kind == "no_inhibition" else Wk,
    }

    camera = SpikeGeneratorGroup(N, input_ids, input_times_ms * ms)
    recurrent_current = "(Exc + Inh)" if kind == "no_gate" else "P*(Exc + Inh)"
    equations = f"""
    dv/dt = (-v + ExtIn + {recurrent_current})/tau : 1
    dExtIn/dt = -ExtIn/tau : 1
    dExc/dt = -Exc/tau : 1
    dInh/dt = -Inh/tau : 1
    incoming_spikes : integer
    P : integer
    """
    neurons = NeuronGroup(
        N, equations, threshold="v > vt",
        reset="v = vr; incoming_spikes = 0",
        refractory=0*ms,
        events={"P_ON": "v > vt",
                "P_OFF": "(timestep(t - lastspike, dt) > timestep(dt, dt) and v <= vt)"},
        method="exact", namespace=params,
    )
    neurons.set_event_schedule("P_ON", when="after_thresholds")
    neurons.run_on_event("P_ON", "P = 1", when="after_thresholds")
    neurons.set_event_schedule("P_OFF", when="groups")
    neurons.run_on_event("P_OFF", "P = 0", when="groups")

    feed = Synapses(camera, neurons, on_pre="ExtIn_post = beta", namespace=params)
    feed.connect(j="i")

    if kind == "synaptic_inhibition":
        on_pre = {
            "pre": "Exc_pre = Wi",
            "pre_2": "Inh_post += P_post * Wk",
        }
    else:
        inhibitory_update = (
            "Inh_post = Wk * incoming_spikes_post" if kind == "no_gate"
            else "Inh_post = P_post * Wk * incoming_spikes_post"
        )
        on_pre = {
            "pre": "incoming_spikes_post += 1; Exc_pre = Wi",
            "pre_2": inhibitory_update,
        }

    recurrent = Synapses(neurons, neurons, on_pre=on_pre, namespace=params)
    recurrent.pre.order = 0
    recurrent.pre_2.order = 1

    pre = np.repeat(np.arange(N), N)
    post = np.tile(np.arange(N), N)
    adjacent = ((pre != post)
                & (np.abs(pre % width - post % width) <= k)
                & (np.abs(pre // width - post // width) <= k))
    recurrent.connect(i=pre[adjacent], j=post[adjacent])

    spikes = SpikeMonitor(neurons)
    states = StateMonitor(
        neurons, ["v", "P", "ExtIn", "Exc", "Inh", "incoming_spikes"],
        record=[center, neighbor], dt=clock_step,
    )
    network = Network(camera, neurons, feed, recurrent, spikes, states)
    network.run(end_ms * ms)

    return {
        "spike_t": np.asarray(spikes.t / ms),
        "spike_i": np.asarray(spikes.i, dtype=int),
        "t": np.asarray(states.t / ms),
        "state": {name: np.asarray(getattr(states, name)).copy()
                  for name in ("v", "P", "ExtIn", "Exc", "Inh", "incoming_spikes")},
        "input_t": np.asarray(input_times_ms),
        "input_i": np.asarray(input_ids),
    }

results = {}
total = len(variants) * len(stimuli)
for run_index, (kind, name) in enumerate(
    ((kind, name) for kind in variants for name in stimuli), start=1
):
    ids, times = stimuli[name]
    results[kind, name] = simulate(kind, ids, times)
    print(f"[{run_index:2d}/{total}] {kind:23s} {name:13s} "
          f"output spikes = {len(results[kind, name]['spike_i']):4d}")

## 3. Quantify persistence and competition

Output spikes are compared on two explicitly defined intervals:

- **Late autonomous activity:** output spikes during 20–60 ms, where no camera input occurs in any protocol. This distinguishes brief responses from sustained output.
- **Sequential replacement:** in the sequential experiment, count spikes from the original center and new neighbor during 10–60 ms (after the second event at 5 ms).

The paper's desired pattern is sustained firing after an isolated input and preferential persistence of the more recently stimulated location. Whether a variant matches those patterns is determined from the measurements, **not assumed**.

In [ ]:
late_counts = np.zeros((len(variants), len(stimuli)), dtype=int)
total_counts = np.zeros_like(late_counts)
for i, kind in enumerate(variants):
    for j, name in enumerate(stimuli):
        result = results[kind, name]
        late_counts[i, j] = np.count_nonzero(result["spike_t"] >= 20.0)
        total_counts[i, j] = len(result["spike_i"])

print("Output spikes: total / late (20–60 ms)")
print(f"{'Variant':23s}" + "".join(f"{name:>18s}" for name in stimuli))
for i, kind in enumerate(variants):
    print(f"{kind:23s}" + "".join(
        f"{total_counts[i,j]:5d} / {late_counts[i,j]:<6d}" for j in range(len(stimuli))
    ))

print("\nSequential replacement: spikes from 10–60 ms")
for kind in variants:
    r = results[kind, "sequential"]
    old_spikes = np.count_nonzero(
        (r["spike_i"] == center) & (r["spike_t"] >= 10))
    new_spikes = np.count_nonzero(
        (r["spike_i"] == neighbor) & (r["spike_t"] >= 10))
    print(f"{kind:23s} old={old_spikes:3d}, new={new_spikes:3d}")

fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
im = ax.imshow(late_counts, aspect="auto", cmap="viridis")
ax.set_xticks(np.arange(len(stimuli)), labels=list(stimuli))
ax.set_yticks(np.arange(len(variants)), labels=list(variants))
ax.set_title("Autonomous output spikes from 20 to 60 ms")
ax.set_xlabel("Input protocol")
ax.set_ylabel("Model variant")
for i in range(len(variants)):
    for j in range(len(stimuli)):
        ax.text(j, i, str(late_counts[i, j]), ha="center", va="center",
                color="white" if late_counts[i, j] < late_counts.max()/2 else "black")
fig.colorbar(im, ax=ax, label="Output spike count")
plt.show()

## 4. Spike rasters: where does activity persist?

The numerical counts above aggregate across neurons. Spike rasters identify *which* neuron sustains activity, distinguishing desirable replacement from indiscriminate or runaway firing.

Each panel uses the same time and neuron axes. Rows correspond to model variants, and columns show the single-event and sequential-neighbor protocols. Red crosses mark camera inputs; the dashed line indicates the final camera event.

In [ ]:
shown = ("single", "sequential")
fig, axes = plt.subplots(len(variants), len(shown), figsize=(13, 12),
                         sharex=True, sharey=True, constrained_layout=True)
for row, kind in enumerate(variants):
    for col, name in enumerate(shown):
        ax = axes[row, col]
        r = results[kind, name]
        ax.scatter(r["spike_t"], r["spike_i"], s=12, alpha=0.8)
        ax.scatter(r["input_t"], r["input_i"], s=60, marker="x",
                   linewidths=1.6, color="tab:red")
        ax.axvline(r["input_t"].max(), color="0.3", linestyle="--", linewidth=1)
        ax.set(xlim=(0, end_ms), ylim=(-0.5, N - 0.5),
               ylabel=f"{kind}\nNeuron ID" if col == 0 else "")
        if row == 0:
            ax.set_title(name.capitalize())
        if row == len(variants)-1:
            ax.set_xlabel("Model time (ms)")
fig.suptitle("ESCM 1.x ablations and counter-free inhibitory candidate", fontsize=14)
plt.show()

## 5. State traces: why does the outcome change?

For the **sequential** protocol, compare the old and new neurons' membrane voltage, persistence gate and inhibitory trace. These traces help distinguish three different failures:

1. A neuron never becomes active: input-response failure.
2. A neuron spikes briefly and then stops: failure of self-sustained activity.
3. An old neuron persists after new neighbor activation: failure of lateral competition.

The counter-free candidate should have an unused counter trace fixed at zero. That is by design and shows the candidate is not indirectly using the old bookkeeping variable.

In [ ]:
fig, axes = plt.subplots(3, len(variants), figsize=(16, 8),
                         sharex=True, constrained_layout=True)
for col, kind in enumerate(variants):
    r = results[kind, "sequential"]
    for row, key in enumerate(("v", "P", "Inh")):
        ax = axes[row, col]
        ax.plot(r["t"], r["state"][key][0], label="Old center")
        ax.plot(r["t"], r["state"][key][1], label="New neighbor")
        ax.axvline(5, color="0.5", linestyle="--", linewidth=1)
        ax.grid(alpha=0.2)
        if row == 0:
            ax.set_title(kind.replace("_", " "), fontsize=10)
            ax.axhline(vt, color="black", linestyle=":", linewidth=1)
        if col == 0:
            ax.set_ylabel(key)
        if row == 2:
            ax.set_xlabel("Time (ms)")
axes[0, 0].legend(loc="upper right", fontsize=8)
fig.suptitle("Sequential stimulation: old versus newly activated neuron", fontsize=14)
plt.show()

## 6. Interpreting the ablations and the next decision

**What would support the hypothesized causal roles?**

- If removing self-excitation eliminates late firing in the **single** condition, self-excitation is necessary for autonomous persistence under these settings.
- If removing inhibition allows both neurons to continue firing in **sequential** stimulation, inhibition is necessary for replacing old activity.
- If removing \(P\) changes responsiveness or causes unwanted activity, the gate has a functional role; it should not be discarded merely because it looks like a software workaround.
- If the counter-free candidate preserves both *single-event persistence* and *sequential replacement*, it is a promising simpler implementation. But its exact spike trains need not match the baseline.

**Limitations.** These are 25-neuron, 60-ms tests at one timestep and parameter set. The neighborhood radius \(k=8\) links every distinct pair of cells, unlike a larger image. No conclusion about physiological realism, scaling, or actual chip feasibility follows from these tests alone.

**Next work package:** If the counter-free candidate fails, examine inhibitory decay and update semantics (assignment versus accumulation) rather than changing every parameter simultaneously. Once the minimal mechanism works, repeat the comparison on the moving-edge stimulus and a fixed slice of the real Directions recording.

**Execution status:** Prepared for execution in the user's Brian2 environment; results are not asserted in advance.